# Bayes’sche Netze – Struktur, CPDs und Inferenz

## Szenario: Judea Pearls Alarm-Netz

In diesem Notebook verwenden wir das klassische **Alarm-Netz von Judea Pearl**, um zu untersuchen, wie Bayes’sche Netze probabilistische Abhängigkeiten zwischen Variablen darstellen.

Das Netz modelliert die Zusammenhänge zwischen:

- einem **Einbruch**,
- einem **Erdbeben**,
- einem **Alarm**,
- sowie den Anrufen von **John** und **Mary**.

Mit Hilfe der Python-Bibliothek `pgmpy` bilden wir die Netzstruktur ab, definieren bedingte Wahrscheinlichkeitsverteilungen (CPDs) und führen probabilistische Inferenz durch.

> **Wichtig:** Das Ziel ist nicht nur, Wahrscheinlichkeiten auszurechnen. Entscheidend ist zu verstehen, wie **Netzstruktur**, **bedingte Wahrscheinlichkeiten** und vorhandene **Evidenz** gemeinsam das Ergebnis einer Inferenz bestimmen.

## Lernziele

1. die **Struktur eines Bayes’schen Netzes** als gerichteten azyklischen Graphen (DAG) darstellen,

2. **Prior-Wahrscheinlichkeiten und bedingte Wahrscheinlichkeitsverteilungen (CPDs)** mit `pgmpy` definieren,

3. erklären, wie **Conditional Probability Tables** aufgebaut sind und wie Kombinationen von Elternvariablen angeordnet werden,

4. mit Hilfe von **Variable Elimination** Wahrscheinlichkeitsanfragen in einem Bayes’schen Netz beantworten,

5. verschiedene Arten der Inferenz unterscheiden, insbesondere:
   - prädiktive Inferenz,
   - diagnostische Inferenz,
   - interkausale Inferenz,

6. erklären, wie neue **Evidenz Wahrscheinlichkeiten verändert**,

7. Ergebnisse probabilistischer Inferenz **inhaltlich interpretieren** und nicht nur numerisch berechnen.

## Ablauf

Das Notebook gliedert sich in drei Schritte:

**1. Netz modellieren**  
Knoten, gerichtete Kanten, Priors und bedingte Wahrscheinlichkeitsverteilungen definieren.

**2. Inferenz durchführen**  
Beobachtete Variablen als Evidenz verwenden und bedingte Wahrscheinlichkeiten berechnen.

**3. Unterschiedliche Inferenzrichtungen vergleichen**  
Untersuchen, wie sich Wahrscheinlichkeiten verändern, wenn von Ursachen zu Wirkungen, von Wirkungen zu Ursachen oder zwischen konkurrierenden Ursachen geschlossen wird.

Eine zentrale Frage des Notebooks lautet:

> **Wie verändert Evidenz unsere Überzeugungen über Variablen, die wir nicht direkt beobachten können?**

# 1. Netz modellieren

Untenstehender Code verwendet python mit der pgmpy Library für Bayes'sche Netze, um das berühmte Alarmbeispiel von Judea Pearl zu implementieren. Machen Sie sich mit dem Code vertraut: 
* Wie legt man die Netzstruktur an?
* Wie legt man die bedingten Wahrscheinlichkeiten fest, insbesondere: in welcher Reohenfolge steht was in den Conditional Probability Tables?
* Wie funktioniert die Inferenz?

In [1]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# 1. Define structure of the Bayes Net with its edges from parents to children
model = DiscreteBayesianNetwork([
    ('Einbruch', 'Alarm'),
    ('Erdbeben', 'Alarm'),
    ('Alarm', 'JohnRuftAn'),
    ('Alarm', 'MaryRuftAn')
])

# 2. Define probabilities
# Unconditioned probabilities (Priors)
# ...variable_card is the cardinality, i.e. the number of states the variable can have
# ...CPD: conditional probability distribution
cpd_einbruch = TabularCPD(variable='Einbruch', variable_card=2,
                           values=[[0.999], [0.001]])  # order is [False, True]
cpd_erdbeben = TabularCPD(variable='Erdbeben', variable_card=2,
                           values=[[0.998], [0.002]])

# Alarm | Einbruch, Erdbeben
cpd_alarm = TabularCPD(
    variable='Alarm', variable_card=2,
    values=[
        [0.999, 0.71, 0.06, 0.05],  # Alarm = False for evidence Einbruch-Erdbeben 0-0, 0-1, 1-0, 1-1
        [0.001, 0.29, 0.94, 0.95]   # Alarm = True (order like above)
    ],
    evidence=['Einbruch', 'Erdbeben'], # pgmpy expects lexicographic order
    evidence_card=[2, 2]
)

# JohnRuftAn | Alarm
cpd_john = TabularCPD(variable='JohnRuftAn', variable_card=2,
                      values=[[0.95, 0.1],   # JohnRuftAn = False for evidence Alarm = 0 resp. 1
                              [0.05, 0.9]],  # JohnRuftAn = True
                      evidence=['Alarm'],
                      evidence_card=[2])

# MaryRuftAn | Alarm
cpd_mary = TabularCPD(variable='MaryRuftAn', variable_card=2,
                      values=[[0.99, 0.3],   # MaryRuftAn = False for evidence Alarm = 0 resp. 1
                              [0.01, 0.7]],  # MaryRuftAn = True
                      evidence=['Alarm'],
                      evidence_card=[2])

# add CPDs to model
model.add_cpds(cpd_einbruch, cpd_erdbeben, cpd_alarm, cpd_john, cpd_mary)

# Check model consistency (e.g., all variables have CPDs, dimensions match, CPDs are normalized)
assert model.check_model()

# 3. Initialize inference engine
inference = VariableElimination(model)

# 4. Compute and print marginals
print("Computed marginals are:")
alarm = inference.query(variables=['Alarm'])
print(alarm)
john = inference.query(variables=['JohnRuftAn'])
print(john)
mary = inference.query(variables=['MaryRuftAn'])
print(mary)

# === Diagnostic Inference ===
print("Inference - Task 1:")
# Given: JohnRuftAn = False, MaryRuftAn = False
# Tbd: Probability for Einbruch
alarm = inference.query(variables=['Alarm'],
                        evidence={'JohnRuftAn': 0, 'MaryRuftAn': 0})
print(alarm)
diagnosis = inference.query(variables=['Einbruch'],
                          evidence={'JohnRuftAn': 0, 'MaryRuftAn': 0})
print("\nDiagnostische Inferenz (p(Einbruch | JohnRuftAn=False, MaryRuftAn=False)):")
print(diagnosis)



Computed marginals are:
+----------+--------------+
| Alarm    |   phi(Alarm) |
+==========+==============+
| Alarm(0) |       0.9975 |
+----------+--------------+
| Alarm(1) |       0.0025 |
+----------+--------------+
+---------------+-------------------+
| JohnRuftAn    |   phi(JohnRuftAn) |
+===============+===================+
| JohnRuftAn(0) |            0.9479 |
+---------------+-------------------+
| JohnRuftAn(1) |            0.0521 |
+---------------+-------------------+
+---------------+-------------------+
| MaryRuftAn    |   phi(MaryRuftAn) |
+===============+===================+
| MaryRuftAn(0) |            0.9883 |
+---------------+-------------------+
| MaryRuftAn(1) |            0.0117 |
+---------------+-------------------+
Inference - Task 1:
+----------+--------------+
| Alarm    |   phi(Alarm) |
+==========+==============+
| Alarm(0) |       0.9999 |
+----------+--------------+
| Alarm(1) |       0.0001 |
+----------+--------------+

Diagnostische Inferenz (p(Einbr

# 2. Inferenz durchführen

Implementieren Sie die Beispiele aus dem Skript für
  
Prädiktive Inferenz...

    d.h. wie hoch ist die Wahrscheinlichkeit, dass John anruft, wenn eingebrochen wurde?


In [2]:
# TODO



# 3. Unterschiedliche Inferenzrichtungen vergleichen
...und interkausale Inferenz:

   d.h. wie hoch ist die Wahrscheinlichkeit, dass eingebrochen wurde, wenn der Alarm läutet?
   Wie verändert sich unsere Einschätzung, wenn wir erfahren, dass es ein Erdbeben gab?

In [3]:
# TODO



Sanity Check: Vergleichen Sie Ihre Ergebnisse mit den Ergebnissen aus den Vorlesungsunterlagen!

*Quellenhinweis*: Bei der Strukturierung und Formattierung dieses Notebooks wurde ChatGPT-5.1 verwendet.